# NB-06_salida_cg_coy_dip

Process Flow SAS: **Salida CG coy_DIP** — `ProcessFlowContainer-ooxfyh1DORx6X162`

In [ ]:
# ========= Parámetros =========
# Variables macro del SAS original. El .egp NO las define (venían del
# entorno SAS): su valor sale de la entrevista B4 o de
# project_config.yaml → run.macro_params, o se inyecta acá
# (celda 'parameters' de papermill).

Mes = None  # &Mes — nadie declaró su valor
Year = None  # &Year — nadie declaró su valor

faltantes = [n for n, v in {"Mes": Mes, "Year": Year}.items() if v is None]
if faltantes:
    raise ValueError(f"Parámetros sin valor: {faltantes}")

In [ ]:
# ========= Celda 1: Configuración =========
import pandas as pd
import numpy as np
import os
import sqlalchemy
import datetime
from pathlib import Path
from sqlalchemy import text

# Conexión a BD — la define el orquestador al ejecutar
engine = sqlalchemy.create_engine(os.environ["SASMIG_DB_URL"], pool_pre_ping=True)
# Sesión de BD del notebook — espejo de la sesión WORK de SAS: las
# tablas temporales #tmp viven en ESTA conexión y mueren al cerrar el
# kernel. AUTOCOMMIT: cada statement commitea, como los pasos de SAS.
work_conn = engine.connect().execution_options(isolation_level="AUTOCOMMIT")

# Logging liviano de resultados — aprobado en la entrevista (Fase 4)
_LOG_PATH = Path("log") / "NB-06_salida_cg_coy_dip.log"
_LOG_PATH.parent.mkdir(parents=True, exist_ok=True)
def _log(label, value=None):
    """Una línea por celda: imprime y persiste. Jamás rompe la corrida."""
    try:
        if hasattr(value, "shape"):
            detail = f"{value.shape[0]} filas x {value.shape[1]} cols"
        elif isinstance(value, int):
            detail = f"{value} filas"
        elif value is None:
            detail = "ok"
        else:
            detail = str(value)
        line = f"[{datetime.datetime.now():%Y-%m-%d %H:%M:%S}] {label}: {detail}"
        print(line)
        with open(_LOG_PATH, "a", encoding="utf-8") as fh:
            fh.write(line + "\n")
    except Exception:
        pass  # el log nunca puede tumbar el notebook
with open(_LOG_PATH, "a", encoding="utf-8") as _fh:
    _fh.write(f"\n=== corrida {datetime.datetime.now():%Y-%m-%d %H:%M:%S} ===\n")


## 5- TCons Mixtas_M_DIP_MOD

Arma la ejecución de gasto de transacciones mixtas (M) del año: asigna entidad, actividad y nivel a la ejecución DIPRES (año actual, año anterior y catastro CGR) y expande la muestra al resto de analíticos, entregando el resultado mensual desagregado y agregado

*confianza: medium · verificador: approve · SAS: PROC SQL CREATE TABLE en cascada (LEFT JOIN + CASE + GROUP BY), DATA steps de DROP/concatenación y PROC SORT NODUPKEY*

In [ ]:
# ========= 5- TCons Mixtas_M_DIP_MOD =========
# /*MODIFICADO JUNIO 2025*/
# /* CREA TABLA DE REDISTRIBUCION DE MUESTRA DE TRANSACCIONES MIXTAS (M) SEGUN ESTRUCTURA, PARA EJECUCION AÑO t */
# /*Ejecucion dipres del periodo, con entidad asignada*/
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_dip_1"))
# Year/Mes interpolados como enteros: un bind mataría la #tmp (sp_prepexec)
sql_ejec_dip_1 = f"""
SELECT T1.*, T2.C_ENTIDAD, T2.ACTIVIDAD, T2.NIVEL
INTO #ejec_dip_1
FROM GOBGENER.dbo.EJEC_DIPRES T1
LEFT JOIN GOBGENER.dbo.CR18_ENTID_DIP_Y T2
       ON T1.AÑO = T2.AÑO
      AND T1.C_PARTIDA = T2.C_PARTIDA
      AND T1.C_CAPITULO = T2.C_CAPITULO
      AND T1.C_PROGRAMA = T2.C_PROGRAMA
WHERE T1.AÑO = {int(Year)} AND T1.MES >= {int(Mes)}
"""
res = work_conn.execute(text(sql_ejec_dip_1))
_log("#ejec_dip_1", res.rowcount)


In [ ]:
# data ejec_dip_1 (DROP= PPTO_INICIAL PPTO_ACTUAL EJEC_ACUM); set ejec_dip_1; run;
work_conn.execute(
    text("ALTER TABLE #ejec_dip_1 DROP COLUMN PPTO_INICIAL, PPTO_ACTUAL, EJEC_ACUM")
)


In [ ]:
# /*Ejecucion dipres del periodo, con entidad asignada EN BASE AL AÑO ANTERIOR (NUEVO)*/
# C_ENTIDAD=. -> IS NULL ; ACTIVIDAD=""/NIVEL="" -> NULL (fila sin match en el LEFT JOIN) o blanco
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_dip_2"))
sql_ejec_dip_2 = """
SELECT T1.SUBSECTOR, T1.MONEDA, T1.AÑO, T1.MES, T1.C_PARTIDA, T1.C_CAPITULO, T1.C_PROGRAMA,
       T1.C_TIPO_CUENTA, T1.C_CUENTA, T1.C_ITEM, T1.C_ASIGNACION, T1.C_ANALITICO,
       T1.N_ANALITICO, T1.DEVENG,
       CASE WHEN T1.C_ENTIDAD IS NULL THEN T2.C_ENTIDAD ELSE T1.C_ENTIDAD END AS C_ENTIDAD,
       CASE WHEN T1.ACTIVIDAD IS NULL OR T1.ACTIVIDAD = '' THEN T2.ACTIVIDAD ELSE T1.ACTIVIDAD END AS ACTIVIDAD,
       CASE WHEN T1.NIVEL IS NULL OR T1.NIVEL = '' THEN T2.NIVEL ELSE T1.NIVEL END AS NIVEL
INTO #ejec_dip_2
FROM #ejec_dip_1 T1
LEFT JOIN GOBGENER.dbo.CR18_ENTID_DIP_Y T2
       ON T1.AÑO - 1 = T2.AÑO
      AND T1.C_PARTIDA = T2.C_PARTIDA
      AND T1.C_CAPITULO = T2.C_CAPITULO
      AND T1.C_PROGRAMA = T2.C_PROGRAMA
"""
res = work_conn.execute(text(sql_ejec_dip_2))
_log("#ejec_dip_2", res.rowcount)


In [ ]:
# /*incorpora entidad a faltantes en clasificación dipres (nuevo)*/
# /*obtiene entidades desde cgr*/
work_conn.execute(text("DROP TABLE IF EXISTS #ent_cgr"))
sql_ent_cgr = """
SELECT T1.C_ENTIDAD AS C_ENTIDAD,
       T1.C_PARTIDA AS C_PARTIDA,
       T1.C_CAPITULO AS C_CAPITULO,
       T1.C_PROGRAMA AS C_PROGRAMA,
       T2.ACTIVIDAD AS ACTIVIDAD,
       T2.NIVEL AS NIVEL
INTO #ent_cgr
FROM GOBGENER.dbo.ENTIDADES T1
LEFT JOIN GOBGENER.dbo.CR18_APERTURA T2 ON T1.C_ENTIDAD = T2.ENTIDAD
"""
res = work_conn.execute(text(sql_ent_cgr))
_log("#ent_cgr", res.rowcount)


In [ ]:
# PROC SORT DATA=ent_cgr OUT=ent_cgr NODUPKEY; BY c_partida c_capitulo c_programa;
# el orden no existe en una tabla SQL: el NODUPKEY se resuelve en pandas y vuelve a la sesión
ent_cgr = pd.read_sql(text("SELECT * FROM #ent_cgr"), work_conn)
ent_cgr = ent_cgr.sort_values(
    ["C_PARTIDA", "C_CAPITULO", "C_PROGRAMA"], na_position="first", kind="mergesort"
).drop_duplicates(subset=["C_PARTIDA", "C_CAPITULO", "C_PROGRAMA"], keep="first")
ent_cgr.to_sql("#ent_cgr", work_conn, if_exists="replace", index=False)
_log("ent_cgr", ent_cgr)


In [ ]:
# /*Ejecucion dipres del periodo, con entidad asignada para faltantes a través de dipres*/
COLS_EJEC_DIP = [
    "SUBSECTOR", "MONEDA", "AÑO", "MES", "C_PARTIDA", "C_CAPITULO", "C_PROGRAMA",
    "C_TIPO_CUENTA", "C_CUENTA", "C_ITEM", "C_ASIGNACION", "C_ANALITICO", "N_ANALITICO",
    "DEVENG", "C_ENTIDAD", "ACTIVIDAD", "NIVEL",
]
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_dip"))
sql_ejec_dip = """
SELECT T1.SUBSECTOR, T1.MONEDA, T1.AÑO, T1.MES, T1.C_PARTIDA, T1.C_CAPITULO, T1.C_PROGRAMA,
       T1.C_TIPO_CUENTA, T1.C_CUENTA, T1.C_ITEM, T1.C_ASIGNACION, T1.C_ANALITICO,
       T1.N_ANALITICO, T1.DEVENG,
       CASE WHEN T1.C_ENTIDAD IS NULL THEN T2.C_ENTIDAD ELSE T1.C_ENTIDAD END AS C_ENTIDAD,
       CASE WHEN T1.ACTIVIDAD IS NULL OR T1.ACTIVIDAD = '' THEN T2.ACTIVIDAD ELSE T1.ACTIVIDAD END AS ACTIVIDAD,
       CASE WHEN T1.NIVEL IS NULL OR T1.NIVEL = '' THEN T2.NIVEL ELSE T1.NIVEL END AS NIVEL
INTO #ejec_dip
FROM #ejec_dip_2 T1
LEFT JOIN #ent_cgr T2
       ON T1.C_PARTIDA = T2.C_PARTIDA
      AND T1.C_CAPITULO = T2.C_CAPITULO
      AND T1.C_PROGRAMA = T2.C_PROGRAMA
"""
res = work_conn.execute(text(sql_ejec_dip))
_log("#ejec_dip", res.rowcount)


In [ ]:
# proc sql; drop table ejec_dip_1,ejec_dip_2,ent_cgr; quit;
for _t in ["#ejec_dip_1", "#ejec_dip_2", "#ent_cgr"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {_t}"))


In [ ]:
# /*TCONS MIXTAS DE LA MUESTRA DIPRES*/
_t2_muestra = ", ".join(f"t2.{c}" for c in COLS_EJEC_DIP)
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_muestra_mixtas_m_prev_dip"))
sql_muestra_prev = f"""
SELECT {_t2_muestra},
       t1.C_SCN AS C_SCN,
       t1.C_CAGENTE AS C_CAGENTE,
       CAST(t1.C_CUP AS char(4)) AS C_CUP,
       SUM(t2.DEVENG / 1000000 * t1.POND) AS DEVENGO
INTO #ejec_muestra_mixtas_m_prev_dip
FROM GOBGENER.dbo.EST_MIXTAS_M t1
LEFT JOIN #ejec_dip t2
       ON t2.C_ENTIDAD = t1.ENTIDAD
      AND t2.C_ASIGNACION = t1.C_ASIGNACION
      AND t2.C_ANALITICO = t1.C_ANALITICO
      AND t2.C_CUENTA = t1.C_CUENTA
      AND t2.C_ITEM = t1.C_ITEM
      AND t2.MES = t1.MES
WHERE t2.C_CUENTA IN ('24','33')
  AND t2.C_ITEM IN ('03','01','06','08','09')
  AND t1.ENTIDAD NOT IN (5601, 10201)
GROUP BY {_t2_muestra}, t1.C_SCN, t1.C_CAGENTE, t1.C_CUP
"""
res = work_conn.execute(text(sql_muestra_prev))
_log("#ejec_muestra_mixtas_m_prev_dip", res.rowcount)


In [ ]:
# DATA EJEC_MUESTRA_MIXTAS_M_DIP (DROP= DEVENG); SET EJEC_MUESTRA_MIXTAS_M_prev_DIP;
COLS_MUESTRA = [c for c in COLS_EJEC_DIP if c != "DEVENG"] + ["C_SCN", "C_CAGENTE", "C_CUP", "DEVENGO"]
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_muestra_mixtas_m_dip"))
sql_muestra = f"""
SELECT {", ".join(COLS_MUESTRA)}
INTO #ejec_muestra_mixtas_m_dip
FROM #ejec_muestra_mixtas_m_prev_dip
"""
res = work_conn.execute(text(sql_muestra))
_log("#ejec_muestra_mixtas_m_dip", res.rowcount)
# PROC SQL; DROP TABLE EJEC_MUESTRA_MIXTAS_M_prev_DIP;
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_muestra_mixtas_m_prev_dip"))


In [ ]:
# /* CREA TABLA DE ANALITICOS DE MIXTAS (M) A EXPANDIR (FUERA DE MUESTRA), SEGUN EJECUCION AÑO t */
# t2.C_ENTIDAD NOT IN (5601,10201,.) -> el faltante de SAS es NULL: NOT IN con NULL da UNKNOWN,
# así que el descarte del faltante va explícito con IS NOT NULL
COLS_CRUCE = COLS_EJEC_DIP + ["ENTIDAD", "Clasif_Tcons"]
_t2_cruce = ", ".join(f"t2.{c}" for c in COLS_EJEC_DIP)
work_conn.execute(text("DROP TABLE IF EXISTS #cruce_ejec_exp_mixtas_m_dip"))
sql_cruce = f"""
SELECT {_t2_cruce},
       t1.ENTIDAD AS ENTIDAD,
       t1.Clasif_Tcons AS Clasif_Tcons
INTO #cruce_ejec_exp_mixtas_m_dip
FROM GOBGENER.dbo.ANALIT_TCONS t1
LEFT JOIN #ejec_dip t2
       ON t2.C_ENTIDAD = t1.ENTIDAD
      AND t2.C_ASIGNACION = t1.C_ASIGNACION
      AND t2.C_ANALITICO = t1.C_ANALITICO
      AND t2.C_CUENTA = t1.C_CUENTA
      AND t2.C_ITEM = t1.C_ITEM
WHERE t1.M_MUESTRA = 0
  AND t2.C_ENTIDAD IS NOT NULL
  AND t2.C_ENTIDAD NOT IN (5601, 10201)
  AND t1.Clasif_Tcons = 'M'
"""
res = work_conn.execute(text(sql_cruce))
_log("#cruce_ejec_exp_mixtas_m_dip", res.rowcount)


In [ ]:
# PROM_MES llega de un nodo anterior (input_datasets) como DataFrame: se materializa en la
# sesión para poder cruzarla server-side en el bloque siguiente
prom_mes.to_sql("#prom_mes", work_conn, if_exists="replace", index=False)
_log("prom_mes", prom_mes)


In [ ]:
# /* CREA TABLA DE REDISTRIBUCION EXPANDIDA SEGUN ESTRUCTURA DE MUESTRA, PARA RESTO DE TRANSACCIONES MIXTAS (M), 5216
# PARA EJECUCION AÑO t */
# WHERE t2.DEVENG<>. -> el faltante de SAS es NULL en SQL Server: IS NOT NULL
_t2_exp = ", ".join(f"t2.{c}" for c in COLS_CRUCE)
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_exp_mixtas_m_prelim_dip"))
sql_exp_prelim = f"""
SELECT {_t2_exp},
       t1.C_SCN AS C_SCN,
       t1.C_CUP AS C_CUP,
       t1.C_CAGENTE AS C_CAGENTE,
       SUM(t2.DEVENG) AS DATO_OR,
       SUM(t2.DEVENG / 1000000 * t1.POND * t3.DATO) AS DEVENGO
INTO #ejec_exp_mixtas_m_prelim_dip
FROM GOBGENER.dbo.EST_EXP_MIXTAS_M t1
LEFT JOIN #cruce_ejec_exp_mixtas_m_dip t2
       ON t2.C_CUENTA = t1.C_CUENTA
      AND t2.C_ITEM = t1.C_ITEM
      AND t2.C_ASIGNACION = t1.C_ASIGNACION
      AND t2.C_ANALITICO = t1.C_ANALITICO
      AND t2.MES = t1.MES
      AND t2.ENTIDAD = t1.ENTIDAD
LEFT JOIN #prom_mes t3
       ON t2.MONEDA = t3.MONEDA
      AND t2.MES = t3.MES
      AND t2.AÑO = t3.AÑO
WHERE t2.DEVENG IS NOT NULL
GROUP BY {_t2_exp}, t1.C_SCN, t1.C_CAGENTE, t1.C_CUP
"""
res = work_conn.execute(text(sql_exp_prelim))
_log("#ejec_exp_mixtas_m_prelim_dip", res.rowcount)


In [ ]:
# DATA EJEC_EXP_MIXTAS_M_DIP (DROP= DEVENG DATO_OR Clasif_Tcons ENTIDAD);
_drop_exp = {"DEVENG", "DATO_OR", "Clasif_Tcons", "ENTIDAD"}
COLS_EXP = [
    c
    for c in COLS_CRUCE + ["C_SCN", "C_CUP", "C_CAGENTE", "DATO_OR", "DEVENGO"]
    if c not in _drop_exp
]
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_exp_mixtas_m_dip"))
sql_exp = f"""
SELECT {", ".join(COLS_EXP)}
INTO #ejec_exp_mixtas_m_dip
FROM #ejec_exp_mixtas_m_prelim_dip
"""
res = work_conn.execute(text(sql_exp))
_log("#ejec_exp_mixtas_m_dip", res.rowcount)
# PROC SQL; DROP TABLE EJEC_EXP_MIXTAS_M_prelim_DIP;
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_exp_mixtas_m_prelim_dip"))


In [ ]:
# /* Crea tabla con la ejecución total de transacciones Mixtas (M), Muestra + Resto */
# el SET de SAS alinea por nombre: la MISMA lista explícita en las dos ramas del UNION ALL
_cols_total = ", ".join(COLS_MUESTRA)
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_mixtas_m_total_dip_2"))
sql_total = f"""
SELECT {_cols_total}
INTO #ejec_mixtas_m_total_dip_2
FROM #ejec_muestra_mixtas_m_dip
UNION ALL
SELECT {_cols_total}
FROM #ejec_exp_mixtas_m_dip
"""
res = work_conn.execute(text(sql_total))
_log("#ejec_mixtas_m_total_dip_2", res.rowcount)


In [ ]:
# /* SALIDA de EJECUCION TOTAL DE TRANSACCIONES MIXTAS (M) para CG coyuntura, DESAGREGADA */
# T-SQL no admite alias en el GROUP BY (CALCULATED TRIM del SAS): la expresión se repite
_trim_case = (
    "CASE WHEN t1.MES IN (1,2,3) THEN 1 "
    "WHEN t1.MES IN (4,5,6) THEN 2 "
    "WHEN t1.MES IN (7,8,9) THEN 3 ELSE 4 END"
)
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_g_mixt_aper_dip"))
sql_aper = f"""
SELECT t1.SUBSECTOR, t1.MONEDA, t1.AÑO,
       {_trim_case} AS [TRIM],
       t1.MES, t1.C_ENTIDAD, t1.C_PARTIDA, t1.C_CAPITULO, t1.C_PROGRAMA, t1.C_TIPO_CUENTA,
       t1.C_CUENTA, t1.C_ITEM, t1.C_ASIGNACION, t1.C_ANALITICO, t1.N_ANALITICO,
       t1.C_SCN, t1.C_CAGENTE, t1.C_CUP,
       SUM(t1.DEVENGO) AS Dato,
       t1.ACTIVIDAD, t1.NIVEL
INTO #ejec_g_mixt_aper_dip
FROM #ejec_mixtas_m_total_dip_2 t1
WHERE t1.C_SCN IN ('D11','P2','D63')
GROUP BY t1.SUBSECTOR, t1.MONEDA, t1.AÑO, {_trim_case}, t1.MES, t1.C_ENTIDAD, t1.C_PARTIDA,
         t1.C_CAPITULO, t1.C_PROGRAMA, t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM,
         t1.C_ASIGNACION, t1.C_ANALITICO, t1.N_ANALITICO, t1.C_SCN, t1.C_CAGENTE, t1.C_CUP,
         t1.ACTIVIDAD, t1.NIVEL
"""
res = work_conn.execute(text(sql_aper))
_log("#ejec_g_mixt_aper_dip", res.rowcount)


In [ ]:
# /*SALIDA AGREGADA*/
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_g_mixt_dip"))
sql_g_mixt = """
SELECT t1.SUBSECTOR, t1.MONEDA, t1.AÑO, t1.[TRIM], t1.MES,
       t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM,
       t1.C_SCN, t1.C_CAGENTE, t1.C_CUP,
       SUM(t1.Dato) AS Dato,
       t1.ACTIVIDAD, t1.NIVEL
INTO #ejec_g_mixt_dip
FROM #ejec_g_mixt_aper_dip t1
GROUP BY t1.SUBSECTOR, t1.MONEDA, t1.AÑO, t1.[TRIM], t1.MES,
         t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM,
         t1.C_SCN, t1.C_CAGENTE, t1.ACTIVIDAD, t1.NIVEL, t1.C_CUP
"""
work_conn.execute(text(sql_g_mixt))
ejec_g_mixt_dip = pd.read_sql(text("SELECT * FROM #ejec_g_mixt_dip"), work_conn)
_log("ejec_g_mixt_dip", ejec_g_mixt_dip)


In [ ]:
# PROC SQL; DROP TABLE ... ; QUIT;  (limpieza final del SAS; los nombres que este flujo
# nunca creó quedan como no-op gracias al IF EXISTS)
for _t in [
    "#est_mixtas_m_prev_dip",
    "#ejec_muestra_mixtas_m_prev_dip",
    "#ejec_muestra_mixtas_m_dip",
    "#analit_tcons_prev_dip",
    "#cruce_ejec_exp_mixtas_m_dip",
    "#ejec_exp_mixtas_m_prelim_dip",
    "#ejec_exp_mixtas_m_dip",
    "#ejec_mixtas_m_total_dip",
    "#est_exp_mixtas_m_prev_dip",
]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {_t}"))


## OPC Aper_CG_coy (solo DIP)

Arma la base mensual del consumo de gobierno de coyuntura con datos DIPRES: clasifica cada gasto según las reglas SCN (analíticos fijos, variables y diferenciados), abre las transferencias en especie de salud según su ratio, incorpora las imputaciones permanentes del año y los ingresos operacionales informados solo en agregado, y entrega la salida desagregada y su total agregado

*confianza: medium · verificador: approve · SAS: PROC SQL (CREATE TABLE / DELETE / UPDATE) + DATA step SET + PROC DATASETS APPEND*

In [ ]:
# ========= OPC Aper_CG_coy (solo DIP) =========
# /* Salida para ANALISIS DESAGREGADO de indicadores GC de consumo de gobierno (coyuntura) */
# /*Salida completa (DATA de CGR + DIPRES (mes t)*/
# Constantes SQL del nodo: el SAS repite la misma lista de C_SCN, el mismo CASE de
# TRIM y el mismo CASE de C_ACT en todas las subconsultas
scn_consumo = "'D1','D11','D12','P2','P3','P72','D63','P1','P11'"
case_trim = """(CASE WHEN t1.MES IN (1,2,3) THEN 1
              WHEN t1.MES IN (4,5,6) THEN 2
              WHEN t1.MES IN (7,8,9) THEN 3
              ELSE 4 END)"""
case_c_act = """(CASE WHEN t1.ACTIVIDAD = 'FFAA' THEN 'AAPP'
              WHEN t1.ACTIVIDAD = 'FFOO' THEN 'AAPP'
              WHEN t1.ACTIVIDAD = 'PREVISION' THEN 'AAPP'
              WHEN t1.ACTIVIDAD = 'FONASA' THEN 'AAPP'
              WHEN t1.ACTIVIDAD = 'CORFO' THEN 'AAPP'
              WHEN t1.ACTIVIDAD = 'CENABAST' THEN 'AAPP'
              WHEN t1.ACTIVIDAD = 'AAPP' THEN 'AAPP'
              WHEN t1.ACTIVIDAD = 'SSPP' THEN 'SSPP'
              WHEN t1.ACTIVIDAD = 'EEPP' THEN 'EEPP' END)"""
# columnas de la salida desagregada, en el orden del SET de SAS
cols_cg = (
    "SUBSECTOR, MONEDA, AÑO, TRIM, MES, C_ENTIDAD, C_PARTIDA, C_CAPITULO, C_PROGRAMA, "
    "C_TIPO_CUENTA, C_CUENTA, C_ITEM, C_ASIGNACION, C_ANALITICO, N_ANALITICO, C_SCN, "
    "C_CAGENTE, C_TRANSACCION, T_TRANSACCION, N_CUENTA, C_ACTIVO, C_CUP, C_ACT, NIVEL, Dato"
)
# columnas de la tabla preliminar (incluye ANALITICO_ y DEVENGO)
cols_prelim = (
    "SUBSECTOR, MONEDA, AÑO, TRIM, MES, C_ENTIDAD, C_PARTIDA, C_CAPITULO, C_PROGRAMA, "
    "C_TIPO_CUENTA, C_CUENTA, C_ITEM, C_ASIGNACION, C_ANALITICO, N_ANALITICO, ANALITICO_, "
    "C_SCN, C_CAGENTE, C_TRANSACCION, T_TRANSACCION, N_CUENTA, C_ACTIVO, C_CUP, C_ACT, NIVEL, DEVENGO"
)


In [ ]:
# /*Analiticos predefinidos */
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_a_prelim_dip_aper"))
sql_a_prelim = f"""
SELECT t1.SUBSECTOR,
       t1.MONEDA,
       t1.AÑO,
       {case_trim} AS TRIM,
       t1.MES,
       t1.C_ENTIDAD,
       t1.C_PARTIDA,
       t1.C_CAPITULO,
       t1.C_PROGRAMA,
       t1.C_TIPO_CUENTA,
       t1.C_CUENTA,
       t1.C_ITEM,
       t1.C_ASIGNACION,
       t1.C_ANALITICO,
       t1.N_ANALITICO,
       (CASE WHEN t2.ANALITICO = 'M' THEN t1.N_ANALITICO ELSE t2.ANALITICO END) AS ANALITICO_,
       t2.C_SCN,
       t2.C_CAGENTE,
       t2.C_TRANSACCION,
       t2.T_TRANSACCION,
       t2.N_CUENTA,
       t2.C_ACTIVO,
       t2.C_CUP,
       {case_c_act} AS C_ACT,
       t1.NIVEL,
       (t1.DEVENG * t3.DATO / 1000000.0) AS DEVENGO
INTO #ejecucion_a_prelim_dip_aper
FROM #ejec_dip t1
LEFT JOIN GOBGENER.dbo.CR18_T_SCN_1 t2
       ON t1.C_TIPO_CUENTA = t2.T_CUENTA
      AND t1.C_CUENTA = t2.C_CUENTA
      AND t1.C_ANALITICO = t2.C_ANALITICO
LEFT JOIN #prom_mes t3
       ON t1.MONEDA = t3.MONEDA AND t1.MES = t3.MES AND t1.AÑO = t3.AÑO
WHERE (t1.C_CUENTA IS NULL OR t1.C_CUENTA NOT IN ('05','13','24','33'))
  AND t2.C_SCN IN ({scn_consumo})
  AND (t1.C_ENTIDAD IS NULL OR t1.C_ENTIDAD NOT IN (5601,10201))
"""
res = work_conn.execute(text(sql_a_prelim))
_log("#ejecucion_a_prelim_dip_aper", res.rowcount)


In [ ]:
# /*nuevo, para eliminar ingresos operacionales que vienen agregados en data dipres. Se incorporan al final del proceso*/
res_item = work_conn.execute(text(
    "DELETE FROM #ejecucion_a_prelim_dip_aper WHERE C_CUENTA = '07' AND C_ITEM IN ('0','00')"
))
# el faltante de caracter en SAS es blanco y entraba al NOT IN; en SQL Server es NULL
res_asig = work_conn.execute(text(
    "DELETE FROM #ejecucion_a_prelim_dip_aper WHERE C_CUENTA = '07' "
    "AND (C_ASIGNACION IS NULL OR C_ASIGNACION NOT IN ('000'))"
))
_log("DELETE #ejecucion_a_prelim_dip_aper ingresos operacionales 07", res_item.rowcount + res_asig.rowcount)


In [ ]:
# /*Estructura para abrir transferencias en especie...*/
# El SAS recrea RATIOS sobre sí misma (SELECT ... FROM RATIOS WHERE C_SCN="D63"):
# se rehace la #tmp usando una tabla puente
work_conn.execute(text("DROP TABLE IF EXISTS #ratios_d63"))
sql_ratios = """
SELECT t1.SUBSECTOR, t1.AÑO, t1.MONEDA, t1.C_ENTIDAD, t1.C_PARTIDA, t1.C_CAPITULO, t1.C_PROGRAMA,
       t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_SCN, t1.C_CAGENTE, t1.C_TRANSACCION, t1.T_TRANSACCION,
       t1.N_CUENTA, t1.C_ACTIVO, t1.C_CUP, t1.C_ACT, t1.NIVEL, t1.Ratio
INTO #ratios_d63
FROM #ratios t1
WHERE t1.C_SCN = 'D63'
"""
res = work_conn.execute(text(sql_ratios))
work_conn.execute(text("DROP TABLE IF EXISTS #ratios"))
work_conn.execute(text("SELECT * INTO #ratios FROM #ratios_d63"))
work_conn.execute(text("DROP TABLE IF EXISTS #ratios_d63"))
_log("#ratios (C_SCN=D63)", res.rowcount)


In [ ]:
# /*apertura transferencias en especie de la salud pública*/
work_conn.execute(text("DROP TABLE IF EXISTS #a_prelim_con_ratio"))
sql_a_prelim_ratio = """
SELECT t1.SUBSECTOR, t1.MONEDA, t1.AÑO, t1.TRIM, t1.MES, t1.C_ENTIDAD, t1.C_PARTIDA, t1.C_CAPITULO,
       t1.C_PROGRAMA, t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM, t1.C_ASIGNACION, t1.C_ANALITICO,
       t1.N_ANALITICO, t1.ANALITICO_, t1.C_SCN, t1.C_CAGENTE, t1.C_TRANSACCION, t1.T_TRANSACCION,
       t1.N_CUENTA, t1.C_ACTIVO, t1.C_CUP, t1.C_ACT, t1.NIVEL, t1.DEVENGO, t2.Ratio
INTO #a_prelim_con_ratio
FROM #ejecucion_a_prelim_dip_aper t1
LEFT JOIN #ratios t2
       ON t1.C_PARTIDA = t2.C_PARTIDA AND t1.C_CAPITULO = t2.C_CAPITULO
      AND t1.C_PROGRAMA = t2.C_PROGRAMA AND t1.C_CUENTA = t2.C_CUENTA AND t1.AÑO = t2.AÑO
"""
res = work_conn.execute(text(sql_a_prelim_ratio))
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_a_prelim_dip_aper"))
work_conn.execute(text("SELECT * INTO #ejecucion_a_prelim_dip_aper FROM #a_prelim_con_ratio"))
work_conn.execute(text("DROP TABLE IF EXISTS #a_prelim_con_ratio"))
_log("#ejecucion_a_prelim_dip_aper con Ratio", res.rowcount)


In [ ]:
# parte en especie: DEVENGO * Ratio, reclasificada a D63 / transferencia en especie
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_a_prelim_dip_ajtesp_ap"))
sql_ajtesp = """
SELECT t1.SUBSECTOR, t1.MONEDA, t1.AÑO, t1.TRIM, t1.MES, t1.C_ENTIDAD, t1.C_PARTIDA, t1.C_CAPITULO,
       t1.C_PROGRAMA, t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM, t1.C_ASIGNACION, t1.C_ANALITICO,
       t1.N_ANALITICO, t1.ANALITICO_,
       'D63' AS C_SCN,
       'S14' AS C_CAGENTE,
       '433' AS C_TRANSACCION,
       'transfesp' AS T_TRANSACCION,
       'IG' AS N_CUENTA,
       t1.C_ACTIVO,
       '272' AS C_CUP,
       t1.C_ACT, t1.NIVEL,
       t1.DEVENGO * t1.Ratio AS DEVENGO
INTO #ejecucion_a_prelim_dip_ajtesp_ap
FROM #ejecucion_a_prelim_dip_aper t1
WHERE t1.Ratio IS NOT NULL
"""
res = work_conn.execute(text(sql_ajtesp))
_log("#ejecucion_a_prelim_dip_ajtesp_ap", res.rowcount)


In [ ]:
# resto no abierto: DEVENGO * (1 - Ratio) cuando hay ratio, DEVENGO completo si no
work_conn.execute(text("DROP TABLE IF EXISTS #a_prelim_neto"))
sql_a_prelim_neto = """
SELECT t1.SUBSECTOR, t1.MONEDA, t1.AÑO, t1.TRIM, t1.MES, t1.C_ENTIDAD, t1.C_PARTIDA, t1.C_CAPITULO,
       t1.C_PROGRAMA, t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM, t1.C_ASIGNACION, t1.C_ANALITICO,
       t1.N_ANALITICO, t1.ANALITICO_, t1.C_SCN, t1.C_CAGENTE, t1.C_TRANSACCION, t1.T_TRANSACCION,
       t1.N_CUENTA, t1.C_ACTIVO, t1.C_CUP, t1.C_ACT, t1.NIVEL,
       (CASE WHEN t1.Ratio IS NOT NULL THEN t1.DEVENGO * (1 - t1.Ratio) ELSE t1.DEVENGO END) AS DEVENGO
INTO #a_prelim_neto
FROM #ejecucion_a_prelim_dip_aper t1
"""
res = work_conn.execute(text(sql_a_prelim_neto))
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_a_prelim_dip_aper"))
work_conn.execute(text("SELECT * INTO #ejecucion_a_prelim_dip_aper FROM #a_prelim_neto"))
work_conn.execute(text("DROP TABLE IF EXISTS #a_prelim_neto"))
_log("#ejecucion_a_prelim_dip_aper neto de transferencias en especie", res.rowcount)


In [ ]:
# PROC DATASETS APPEND FORCE: alinea por nombre, columnas explícitas
# /*fin apertura de transf en especie de salud pública*/
sql_append_ajtesp = f"""
INSERT INTO #ejecucion_a_prelim_dip_aper ({cols_prelim})
SELECT {cols_prelim} FROM #ejecucion_a_prelim_dip_ajtesp_ap
"""
res = work_conn.execute(text(sql_append_ajtesp))
_log("APPEND #ejecucion_a_prelim_dip_ajtesp_ap -> #ejecucion_a_prelim_dip_aper", res.rowcount)


In [ ]:
res_ent = work_conn.execute(text(
    "DELETE FROM #ejecucion_a_prelim_dip_aper WHERE C_ENTIDAD IN (10201)"
))
res_32 = work_conn.execute(text(
    "DELETE FROM #ejecucion_a_prelim_dip_aper WHERE C_CUENTA = '32' "
    "AND C_ANALITICO IN ('04001000','04002000')"
))
# /*Analiticos predefinidos: AJUSTE COYUNTURA - se elimina de indicador transacción CORFO
#   que se desagrega anualmente entre ventas y D45 */
res_corfo = work_conn.execute(text(
    "DELETE FROM #ejecucion_a_prelim_dip_aper WHERE C_ENTIDAD IN (3801) AND C_CUENTA = '06' "
    "AND C_ITEM = '01' AND C_ASIGNACION IN ('00','000') AND C_ANALITICO IN ('01000000')"
))
_log("DELETE #ejecucion_a_prelim_dip_aper ajustes", res_ent.rowcount + res_32.rowcount + res_corfo.rowcount)


In [ ]:
# /*Analiticos predefinidos (sigue). BASE DESGREGADA*/
# El GROUP BY del PROC SQL no lleva funciones de resumen (DEVENGO va sin agregar):
# SAS lo transforma en ORDER BY y NO colapsa filas -> acá va sin GROUP BY
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_a_dip_aper"))
sql_a_dip_aper = """
SELECT t1.SUBSECTOR, t1.MONEDA, t1.AÑO, t1.TRIM, t1.MES, t1.C_ENTIDAD, t1.C_PARTIDA, t1.C_CAPITULO,
       t1.C_PROGRAMA, t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM, t1.C_ASIGNACION, t1.C_ANALITICO,
       t1.N_ANALITICO, t1.C_SCN, t1.C_CAGENTE, t1.C_TRANSACCION, t1.T_TRANSACCION, t1.N_CUENTA,
       t1.C_ACTIVO, t1.C_CUP, t1.C_ACT, t1.NIVEL,
       t1.DEVENGO AS Dato
INTO #ejecucion_a_dip_aper
FROM #ejecucion_a_prelim_dip_aper t1
"""
res = work_conn.execute(text(sql_a_dip_aper))
_log("#ejecucion_a_dip_aper", res.rowcount)


In [ ]:
# /*DIPRES:Analiticos FIJOS CUENTA 32 */
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_b_dip_aper"))
sql_b = f"""
SELECT t1.SUBSECTOR, t1.MONEDA, t1.AÑO,
       {case_trim} AS TRIM,
       t1.MES, t1.C_ENTIDAD, t1.C_PARTIDA, t1.C_CAPITULO, t1.C_PROGRAMA, t1.C_TIPO_CUENTA,
       t1.C_CUENTA, t1.C_ITEM, t1.C_ASIGNACION, t1.C_ANALITICO, t1.N_ANALITICO,
       t2.C_SCN, t2.C_CAGENTE, t2.C_TRANSACCION, t2.T_TRANSACCION, t2.N_CUENTA, t2.C_ACTIVO, t2.C_CUP,
       {case_c_act} AS C_ACT,
       t1.NIVEL,
       (t1.DEVENG * t3.DATO / 1000000.0) AS Dato
INTO #ejecucion_b_dip_aper
FROM #ejec_dip t1
LEFT JOIN GOBGENER.dbo.CR18_T_SCN_2 t2
       ON t1.C_PARTIDA = t2.C_PARTIDA AND t1.C_CAPITULO = t2.C_CAPITULO AND t1.C_PROGRAMA = t2.C_PROGRAMA
      AND t1.C_TIPO_CUENTA = t2.T_CUENTA AND t1.C_CUENTA = t2.C_CUENTA AND t1.C_ITEM = t2.C_ITEM
      AND t1.C_ASIGNACION = t2.C_ASIGNACION AND t1.C_ANALITICO = t2.C_ANALITICO AND t1.C_ENTIDAD = t2.ENTIDAD
LEFT JOIN #prom_mes t3
       ON t1.MONEDA = t3.MONEDA AND t1.MES = t3.MES AND t1.AÑO = t3.AÑO
WHERE t1.C_CUENTA = '32'
  AND t2.C_SCN IN ({scn_consumo})
  AND t1.C_ANALITICO IN ('04001000','04002000')
  AND (t1.C_ENTIDAD IS NULL OR t1.C_ENTIDAD NOT IN (5601,10201))
"""
work_conn.execute(text(sql_b))
res = work_conn.execute(text("DELETE FROM #ejecucion_b_dip_aper WHERE C_ENTIDAD IN (10201)"))
_log("DELETE #ejecucion_b_dip_aper (10201)", res.rowcount)


In [ ]:
# /*analíticos variables*/
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_c_dip_aper"))
sql_c = f"""
SELECT t1.SUBSECTOR, t1.MONEDA, t1.AÑO,
       {case_trim} AS TRIM,
       t1.MES, t1.C_ENTIDAD, t1.C_PARTIDA, t1.C_CAPITULO, t1.C_PROGRAMA, t1.C_TIPO_CUENTA,
       t1.C_CUENTA, t1.C_ITEM, t1.C_ASIGNACION, t1.C_ANALITICO, t1.N_ANALITICO,
       t2.C_SCN, t2.C_CAGENTE, t2.C_TRANSACCION, t2.T_TRANSACCION, t2.N_CUENTA, t2.C_ACTIVO, t2.C_CUP,
       {case_c_act} AS C_ACT,
       t1.NIVEL,
       (t1.DEVENG * t3.DATO / 1000000.0) AS Dato
INTO #ejecucion_c_dip_aper
FROM #ejec_dip t1
LEFT JOIN GOBGENER.dbo.CR18_T_SCN_2 t2
       ON t1.C_PARTIDA = t2.C_PARTIDA AND t1.C_CAPITULO = t2.C_CAPITULO AND t1.C_PROGRAMA = t2.C_PROGRAMA
      AND t1.C_TIPO_CUENTA = t2.T_CUENTA AND t1.C_CUENTA = t2.C_CUENTA AND t1.C_ITEM = t2.C_ITEM
      AND t1.C_ASIGNACION = t2.C_ASIGNACION AND t1.C_ANALITICO = t2.C_ANALITICO AND t1.C_ENTIDAD = t2.ENTIDAD
LEFT JOIN #prom_mes t3
       ON t1.MONEDA = t3.MONEDA AND t1.MES = t3.MES AND t1.AÑO = t3.AÑO
WHERE t1.C_CUENTA IN ('05','13','24','33')
  AND t2.C_SCN IN ({scn_consumo})
  AND (t2.OBS IS NULL OR t2.OBS NOT IN ('CR18_difcoy','CR18_difact'))
  AND (t1.C_ENTIDAD IS NULL OR t1.C_ENTIDAD NOT IN (5601,10201))
  AND (t2.C_SCN IS NULL OR t2.C_SCN NOT IN ('TC'))
"""
work_conn.execute(text(sql_c))
res = work_conn.execute(text("DELETE FROM #ejecucion_c_dip_aper WHERE C_ENTIDAD IN (10201)"))
_log("DELETE #ejecucion_c_dip_aper (10201)", res.rowcount)


In [ ]:
# /* Transacciones diferenciadas para coyuntura */
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_d_dip_aper"))
sql_d = f"""
SELECT t1.SUBSECTOR, t1.MONEDA, t1.AÑO,
       {case_trim} AS TRIM,
       t1.MES, t1.C_ENTIDAD, t1.C_PARTIDA, t1.C_CAPITULO, t1.C_PROGRAMA, t1.C_TIPO_CUENTA,
       t1.C_CUENTA, t1.C_ITEM, t1.C_ASIGNACION, t1.C_ANALITICO, t1.N_ANALITICO,
       t2.C_ACTIVO AS C_SCN,
       t2.C_CAGENTE, t2.C_TRANSACCION, t2.T_TRANSACCION, t2.N_CUENTA, t2.C_ACTIVO, t2.C_CUP,
       {case_c_act} AS C_ACT,
       t1.NIVEL,
       (t1.DEVENG * t3.DATO / 1000000.0) AS Dato
INTO #ejecucion_d_dip_aper
FROM #ejec_dip t1
LEFT JOIN GOBGENER.dbo.CR18_T_SCN_2 t2
       ON t1.C_PARTIDA = t2.C_PARTIDA AND t1.C_CAPITULO = t2.C_CAPITULO AND t1.C_PROGRAMA = t2.C_PROGRAMA
      AND t1.C_TIPO_CUENTA = t2.T_CUENTA AND t1.C_CUENTA = t2.C_CUENTA AND t1.C_ITEM = t2.C_ITEM
      AND t1.C_ASIGNACION = t2.C_ASIGNACION AND t1.C_ANALITICO = t2.C_ANALITICO AND t1.C_ENTIDAD = t2.ENTIDAD
LEFT JOIN #prom_mes t3
       ON t1.MONEDA = t3.MONEDA AND t1.MES = t3.MES AND t1.AÑO = t3.AÑO
WHERE t2.OBS IN ('CR18_difcoy')
  AND (t1.C_ENTIDAD IS NULL OR t1.C_ENTIDAD NOT IN (5601,10201))
  AND t2.C_ACTIVO IN ({scn_consumo})
  AND (t2.C_SCN IS NULL OR t2.C_SCN NOT IN ('TC'))
"""
res = work_conn.execute(text(sql_d))
_log("#ejecucion_d_dip_aper", res.rowcount)


In [ ]:
# /* Transacciones diferenciadas por actividad */
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_f_dip_aper"))
sql_f = f"""
SELECT t1.SUBSECTOR, t1.MONEDA, t1.AÑO,
       {case_trim} AS TRIM,
       t1.MES, t1.C_ENTIDAD, t1.C_PARTIDA, t1.C_CAPITULO, t1.C_PROGRAMA, t1.C_TIPO_CUENTA,
       t1.C_CUENTA, t1.C_ITEM, t1.C_ASIGNACION, t1.C_ANALITICO, t1.N_ANALITICO,
       t2.C_SCN, t2.C_CAGENTE, t2.C_TRANSACCION, t2.T_TRANSACCION, t2.N_CUENTA, t2.C_ACTIVO, t2.C_CUP,
       t2.C_CUP AS C_ACT,
       t1.NIVEL,
       (t1.DEVENG * t3.DATO / 1000000.0) AS Dato
INTO #ejecucion_f_dip_aper
FROM #ejec_dip t1
LEFT JOIN GOBGENER.dbo.CR18_T_SCN_2 t2
       ON t1.C_PARTIDA = t2.C_PARTIDA AND t1.C_CAPITULO = t2.C_CAPITULO AND t1.C_PROGRAMA = t2.C_PROGRAMA
      AND t1.C_TIPO_CUENTA = t2.T_CUENTA AND t1.C_CUENTA = t2.C_CUENTA AND t1.C_ITEM = t2.C_ITEM
      AND t1.C_ASIGNACION = t2.C_ASIGNACION AND t1.C_ANALITICO = t2.C_ANALITICO AND t1.C_ENTIDAD = t2.ENTIDAD
LEFT JOIN #prom_mes t3
       ON t1.MONEDA = t3.MONEDA AND t1.MES = t3.MES AND t1.AÑO = t3.AÑO
WHERE t2.OBS IN ('CR18_difact')
  AND (t1.C_ENTIDAD IS NULL OR t1.C_ENTIDAD NOT IN (5601,10201))
  AND (t2.C_SCN IS NULL OR t2.C_SCN NOT IN ('TC'))
  AND t2.C_SCN IN ({scn_consumo})
"""
res = work_conn.execute(text(sql_f))
_log("#ejecucion_f_dip_aper", res.rowcount)


In [ ]:
# /*FORMATEA TABLA DE TCONS, PARA UNIRLA A LAS OTRAS*/
# DATA step + UPDATE de C_ACT + filtro por C_SCN, resueltos en un solo SELECT
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_g_dip_aper"))
sql_g = f"""
SELECT SUBSECTOR, MONEDA, AÑO, TRIM, MES, C_ENTIDAD, C_PARTIDA, C_CAPITULO, C_PROGRAMA,
       C_TIPO_CUENTA, C_CUENTA, C_ITEM, C_ASIGNACION, C_ANALITICO, N_ANALITICO, C_SCN,
       C_CAGENTE, C_TRANSACCION, T_TRANSACCION, N_CUENTA,
       'TC' AS C_ACTIVO,
       C_CUP,
       (CASE WHEN ACTIVIDAD IN ('FFAA','FFOO','PREVISION','FONASA','CORFO','CENABAST') THEN 'AAPP'
             ELSE ACTIVIDAD END) AS C_ACT,
       NIVEL, Dato
INTO #ejecucion_g_dip_aper
FROM #ejec_g_mixt_aper_dip
WHERE C_SCN IN ({scn_consumo})
"""
res = work_conn.execute(text(sql_g))
_log("#ejecucion_g_dip_aper", res.rowcount)


In [ ]:
# /*UNE TABLAS de Salida total de ejecución CG coy de data DIPRES*/
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_cg_dip_aper_prev"))
sql_union_prev = f"""
SELECT {cols_cg} INTO #ejecucion_cg_dip_aper_prev FROM #ejecucion_a_dip_aper
UNION ALL SELECT {cols_cg} FROM #ejecucion_b_dip_aper
UNION ALL SELECT {cols_cg} FROM #ejecucion_c_dip_aper
UNION ALL SELECT {cols_cg} FROM #ejecucion_d_dip_aper
UNION ALL SELECT {cols_cg} FROM #ejecucion_g_dip_aper
UNION ALL SELECT {cols_cg} FROM #ejecucion_f_dip_aper
"""
res = work_conn.execute(text(sql_union_prev))
_log("#ejecucion_cg_dip_aper_prev", res.rowcount)


In [ ]:
for _t in [
    "#ejecucion_a_prelim_dip_aper",
    "#ejecucion_a_dip_aper",
    "#ejecucion_b_dip_aper",
    "#ejecucion_c_dip_aper",
    "#ejecucion_d_dip_aper",
    "#ejecucion_g_dip_aper",
    "#ejecucion_f_dip_aper",
]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {_t}"))
res = work_conn.execute(text("UPDATE #ejecucion_cg_dip_aper_prev SET N_CUENTA = 'DIPRES'"))
_log("UPDATE #ejecucion_cg_dip_aper_prev N_CUENTA='DIPRES'", res.rowcount)


In [ ]:
# /*Incorporacion de imputaciones permanentes*/
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_cg_dip_aper_imput_perm"))
sql_imput_perm = f"""
SELECT t1.SUBSECTOR, t1.MONEDA, t1.AÑO, t1.TRIM, t1.MES, t1.C_ENTIDAD, t1.C_PARTIDA, t1.C_CAPITULO,
       t1.C_PROGRAMA, t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM, t1.C_ASIGNACION, t1.C_ANALITICO,
       t1.N_ANALITICO, t1.C_SCN, t1.C_CAGENTE, t1.C_TRANSACCION, t1.T_TRANSACCION, t1.N_CUENTA,
       t1.C_ACTIVO, t1.C_CUP,
       (CASE WHEN t1.C_ACT = 'FFAA' THEN 'AAPP'
             WHEN t1.C_ACT = 'FFOO' THEN 'AAPP'
             WHEN t1.C_ACT = 'PREVISION' THEN 'AAPP'
             WHEN t1.C_ACT = 'FONASA' THEN 'AAPP'
             WHEN t1.C_ACT = 'CORFO' THEN 'AAPP'
             WHEN t1.C_ACT = 'CENABAST' THEN 'AAPP'
             WHEN t1.C_ACT = 'AAPP' THEN 'AAPP'
             WHEN t1.C_ACT = 'SSPP' THEN 'SSPP'
             WHEN t1.C_ACT = 'EEPP' THEN 'EEPP' END) AS C_ACT,
       t1.NIVEL, t1.Dato
INTO #ejec_cg_dip_aper_imput_perm
FROM GOBGENER.dbo.IMPUT_GC_DIP_APER t1
WHERE t1.AÑO = {int(Year)}
  AND t1.C_SCN IN ({scn_consumo})
"""
res = work_conn.execute(text(sql_imput_perm))
_log("#ejec_cg_dip_aper_imput_perm", res.rowcount)


In [ ]:
# /*UNE TABLAS*/
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_cg_dip_aper"))
sql_union_aper = f"""
SELECT {cols_cg} INTO #ejecucion_cg_dip_aper FROM #ejecucion_cg_dip_aper_prev
UNION ALL SELECT {cols_cg} FROM #ejec_cg_dip_aper_imput_perm
"""
res = work_conn.execute(text(sql_union_aper))
_log("#ejecucion_cg_dip_aper", res.rowcount)


In [ ]:
# /*NUEVO: IMPUTA INGRESOS OPERACIONALES PARA LAS ENTIDADES QUE LO INFORMAN AGREGADO EN DIPRES, ASÍ SE MEJORA EL INDICADOR DE VENTAS*/
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_a_dip_07"))
sql_a_dip_07 = f"""
SELECT t1.SUBSECTOR, t1.MONEDA, t1.AÑO,
       {case_trim} AS TRIM,
       t1.MES, t1.C_ENTIDAD, t1.C_PARTIDA, t1.C_CAPITULO, t1.C_PROGRAMA, t1.C_TIPO_CUENTA,
       t1.C_CUENTA, t1.C_ITEM, t1.C_ASIGNACION, t1.C_ANALITICO, t1.N_ANALITICO,
       t2.C_SCN, t2.C_CAGENTE, t2.C_TRANSACCION, t2.T_TRANSACCION, t2.N_CUENTA, t2.C_ACTIVO, t2.C_CUP,
       {case_c_act} AS C_ACT,
       t1.NIVEL,
       (t1.DEVENG * t3.DATO / 1000000.0) AS DEVENGO
INTO #ejecucion_a_dip_07
FROM #ejec_dip t1
LEFT JOIN GOBGENER.dbo.CR18_T_SCN_1 t2
       ON t1.C_TIPO_CUENTA = t2.T_CUENTA AND t1.C_CUENTA = t2.C_CUENTA AND t1.C_ANALITICO = t2.C_ANALITICO
LEFT JOIN #prom_mes t3
       ON t1.MONEDA = t3.MONEDA AND t1.MES = t3.MES AND t1.AÑO = t3.AÑO
WHERE t1.C_CUENTA IN ('07')
  AND t2.C_SCN IN ({scn_consumo})
"""
res = work_conn.execute(text(sql_a_dip_07))
_log("#ejecucion_a_dip_07", res.rowcount)


In [ ]:
# /*Verifica que entidades informan solo el agregado para filtrarlas*/
work_conn.execute(text("DROP TABLE IF EXISTS #cuenta_item_07"))
sql_cuenta_item_07 = """
SELECT t1.SUBSECTOR, t1.MONEDA, t1.AÑO, t1.TRIM, t1.MES, t1.C_ENTIDAD, t1.C_PARTIDA, t1.C_CAPITULO,
       t1.C_PROGRAMA, t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_CUP,
       COUNT(t1.DEVENGO) AS id
INTO #cuenta_item_07
FROM #ejecucion_a_dip_07 t1
GROUP BY t1.SUBSECTOR, t1.MONEDA, t1.AÑO, t1.TRIM, t1.MES, t1.C_ENTIDAD, t1.C_PARTIDA, t1.C_CAPITULO,
         t1.C_PROGRAMA, t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_CUP
"""
res = work_conn.execute(text(sql_cuenta_item_07))
_log("#cuenta_item_07", res.rowcount)


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_dip_07"))
sql_dip_07 = """
SELECT t1.SUBSECTOR, t1.MONEDA, t1.AÑO, t1.TRIM, t1.MES, t1.C_ENTIDAD, t1.C_PARTIDA, t1.C_CAPITULO,
       t1.C_PROGRAMA, t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM, t1.C_ASIGNACION, t1.C_ANALITICO,
       t1.N_ANALITICO, t1.C_SCN, t1.C_CAGENTE, t1.C_TRANSACCION, t1.T_TRANSACCION,
       'DIPRES' AS N_CUENTA,
       t1.C_ACTIVO, t1.C_CUP, t1.C_ACT, t1.NIVEL,
       t1.DEVENGO AS Dato
INTO #ejecucion_dip_07
FROM #ejecucion_a_dip_07 t1
LEFT JOIN #cuenta_item_07 t2
       ON t1.MONEDA = t2.MONEDA AND t1.C_ENTIDAD = t2.C_ENTIDAD AND t1.C_PARTIDA = t2.C_PARTIDA
      AND t1.C_CAPITULO = t2.C_CAPITULO AND t1.C_PROGRAMA = t2.C_PROGRAMA
      AND t1.C_TIPO_CUENTA = t2.C_TIPO_CUENTA AND t1.C_CUENTA = t2.C_CUENTA AND t1.MES = t2.MES
WHERE t2.id = 1
"""
res = work_conn.execute(text(sql_dip_07))
_log("#ejecucion_dip_07", res.rowcount)


In [ ]:
# /*una imputacion de ingresos operacionales*/
sql_append_07 = f"""
INSERT INTO #ejecucion_cg_dip_aper ({cols_cg})
SELECT {cols_cg} FROM #ejecucion_dip_07
"""
res = work_conn.execute(text(sql_append_07))
_log("APPEND #ejecucion_dip_07 -> #ejecucion_cg_dip_aper", res.rowcount)


In [ ]:
res = work_conn.execute(text("DELETE FROM #ejecucion_cg_dip_aper WHERE NIVEL = 'EXCLUIDO'"))
_log("DELETE #ejecucion_cg_dip_aper (NIVEL EXCLUIDO)", res.rowcount)


In [ ]:
# /*GENERA SALIDA AGREGADA*/
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_cg_dip"))
sql_cg_dip = """
SELECT t1.SUBSECTOR, t1.MONEDA, t1.AÑO, t1.TRIM, t1.MES, t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM,
       t1.C_SCN, t1.C_CAGENTE, t1.C_TRANSACCION, t1.T_TRANSACCION, t1.N_CUENTA, t1.C_ACTIVO,
       t1.C_CUP, t1.C_ACT, t1.NIVEL,
       SUM(t1.Dato) AS Dato
INTO #ejecucion_cg_dip
FROM #ejecucion_cg_dip_aper t1
GROUP BY t1.SUBSECTOR, t1.MONEDA, t1.AÑO, t1.TRIM, t1.MES, t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM,
         t1.C_SCN, t1.C_CAGENTE, t1.C_TRANSACCION, t1.T_TRANSACCION, t1.N_CUENTA, t1.C_ACTIVO,
         t1.C_CUP, t1.C_ACT, t1.NIVEL
"""
res = work_conn.execute(text(sql_cg_dip))
_log("#ejecucion_cg_dip", res.rowcount)


In [ ]:
# /*EXPORTAR A EXCEL LAS TABLAS: EJECUCION_CG_DIP_aper Y EJECUCION_CG_DIP*/
# (el SAS solo lo deja anotado: no hay PROC EXPORT en este nodo)
for _t in [
    "#ejec_cg_dip_aper_imput_perm",
    "#cr18_apertura_pacapr",
    "#cuenta_item_07",
    "#ejecucion_a_dip_07",
    "#ejecucion_a_prelim_dip_ajtesp_ap",
    "#ejec_dip",
    "#ejecucion_dip_07",
    "#ejecucion_cg_dip_aper_prev",
]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {_t}"))
